# 第2回宿題（実装）：架空のセキュリティ監視FAQをLoRAで学習

公開リポジトリ：https://github.com/masahikoyr/lecture-ai-engineering-2026

講座公式 `notebooks/lora_format_tuning.ipynb` を基に、独自FAQと評価質問に置換。
モデル・revision・ライブラリの固定版・学習設定は講座のままです。

1. Colabの「ファイル → ドライブにコピーを保存」でコピーする。
2. 「ランタイム → ランタイムのタイプを変更 → T4 GPU」を選択する。
3. 上から全セルを実行する（初回ダウンロードと学習で十数分以上かかる場合があります）。
4. 最終セルが `AIEngineering2026_Lec02_results.zip` を作成し、ダウンロードを開始する。
5. ZIP内の `answer.md` を原文と照合し、本人確認欄を記入して公開forkの `lectures/02/` に配置する。

Gemini APIキーは不要。公開モデルをダウンロードしてColabで学習します。
このノートブックの結果は**未実行**です。コード内の期待値やテスト値を実測として提出しません。
同じセッションで学習セルを繰り返さず、やり直すときはセッションを再起動して先頭から実行します。

FAQはすべて架空の訓練規則です。実際の顧客情報・連絡先・ログは入力しないでください。


## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")
if not gpu_info.strip():
    raise RuntimeError("T4 GPUを選択してから、先頭から実行してください。")


In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。
- `num_train_epochs: 20` は、学習データ（14 件）を 20 周する設定です。1 周が 2 ステップなので合計 40 ステップで、T4 で学習が 4〜5 分に収まります。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。

**見るところ**: 設定の JSON が表示されます。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/soc_faq_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/soc_faq_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
EXECUTED_TRAINING = False


## 2. 独自FAQと教師データ

FAQは5項目：不審メール報告、ログ保全、重要度、隔離承認、引継ぎ。
根拠なしの質問は、FAQにない具体的な電話番号、保存日数、製品名、実行コマンド等を聞く。
train=14 / valid=7 / test=21。質問文は分割間で重複させない。
JSONの `answer` と `needs_review` は講座の設計を維持する。
以下は `lectures/02/soc_faq.py` と同じソースを埋め込んでおり、外部ファイルのアップロードは不要。


In [ ]:
"""第2回実装課題: 実在組織を含まない、架空のセキュリティ監視FAQ。"""
import json

FAQ = {
    "report": "不審メールは開かず、訓練窓口へ報告します。",
    "preserve": "調査ログは原本を保存し、コピーで分析します。",
    "priority": "重要度は影響と緊急度で判断します。",
    "approval": "端末隔離は責任者の承認後に実施します。",
    "handover": "引継ぎには事象と対応状況を記録します。",
}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())
GROUNDED = {
    "report": (["不審メールはどう扱いますか", "怪しいメールを受け取ったら？", "不審なメールの扱い方は？", "不審メールの対応手順は？"], ["開かず", "訓練窓口", "報告"]),
    "preserve": (["調査ログはどう扱いますか", "ログ分析では原本を使いますか", "分析するログの保管方法は？", "ログの原本とコピーはどう使いますか"], ["原本", "保存", "コピー"]),
    "priority": (["重要度は何で判断しますか", "事象の優先度は何を見て決めますか", "重要度の判定基準は？", "重要度を判断する二つの観点は？"], ["影響", "緊急度"]),
    "approval": (["端末隔離には何が必要ですか", "端末を隔離する前に必要な手続きは？", "端末隔離はいつ実施しますか", "隔離を実行するには誰の承認が必要ですか"], ["責任者", "承認"]),
    "handover": (["引継ぎには何を記録しますか", "担当交代時に残す情報は？", "次の担当者へ何を伝えますか", "引継ぎ記録に必要な二つの内容は？"], ["事象", "対応状況"]),
}
# FAQに書かれていない「近い話題」を用意し、知っていそうでも断定しないか確認する。
UNGROUNDED = [
    "訓練窓口の電話番号は何番ですか", "調査ログの保存期間は何日ですか",
    "重要度の判定には何段階ありますか", "隔離の承認者が不在ならどうしますか",
    "不審メールは何分以内に報告しますか", "ログの保存先パスは何ですか",
    "引継ぎの実施時刻は何時ですか", "承認後に隔離する際の製品名は何ですか",
]
TEST_EXTRA_GROUNDED = {
    "report": ["疑わしいメールは開いてよいですか", "怪しいメールの報告先と扱いを教えて"],
    "preserve": ["分析時はログのどちらを使いますか", "調査で原本ログはどうしますか"],
    "priority": ["影響と何を見て重要度を決めますか", "重要度評価で見る要素を挙げて"],
    "approval": ["責任者が承認する前に隔離しますか", "端末隔離前に得るものは何ですか"],
    "handover": ["引継ぎメモの記録内容を教えて", "担当を交代するときに残す内容は何ですか"],
}
TEST_EXTRA_UNGROUNDED = [
    "ログ分析に使う製品の名前は何ですか", "重大事象の報告期限は何分ですか",
    "端末隔離を実行するコマンドを教えて", "引継ぎ記録の保存先URLはどこですか",
]
UNGROUNDED_ANSWER = "この FAQ には記載がありません。責任者に確認が必要です。"
KNOWLEDGE_QUESTIONS = [
    "訓練窓口の担当者の氏名は何ですか",
    "重大事象で外部機関へ通報する期限は何分ですか",
    "ログを暗号化するときの暗号方式は何ですか",
    "端末隔離には何が必要ですか",
]
KNOWLEDGE_IN_FAQ = {q: i == 3 for i, q in enumerate(KNOWLEDGE_QUESTIONS)}


def make_example(question, answer, needs_review):
    return {"question": question, "faq": FAQ_TEXT,
            "target": json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False),
            "needs_review": needs_review}


def make_splits():
    train, valid, test, gold = [], [], [], {}
    for key, (questions, must_include) in GROUNDED.items():
        for q in questions[:2]: train.append(make_example(q, FAQ[key], False))
        valid.append(make_example(questions[2], FAQ[key], False))
        for q in questions[3:] + TEST_EXTRA_GROUNDED[key]:
            test.append(make_example(q, FAQ[key], False))
            gold[q] = {"needs_review": False, "must_include": must_include}
    for i, q in enumerate(UNGROUNDED):
        bucket = (train, train, valid, test)[i % 4]
        bucket.append(make_example(q, UNGROUNDED_ANSWER, True))
        if bucket is test:
            gold[q] = {"needs_review": True, "must_include": ["確認"]}
    for q in TEST_EXTRA_UNGROUNDED:
        test.append(make_example(q, UNGROUNDED_ANSWER, True))
        gold[q] = {"needs_review": True, "must_include": ["確認"]}
    questions = [e["question"] for e in train + valid + test]
    assert len(questions) == len(set(questions)), "分割間に質問の重複があります"
    return train, valid, test, gold

train, valid, test, gold = make_splits()
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
for name, examples in [("train", train), ("valid", valid), ("test", test)]:
    (WORK / f"{name}.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in examples), encoding="utf-8")
(WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")
print("データ例:", train[0])


## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の 21 問に答えさせて採点します。

プロンプト（`SYSTEM`）には「必ず JSON 1 行で、`answer` と `needs_review` を返す」と書いてあります。つまり **形式は指示済み** です。このモデルがその指示を守れるかどうかが、学習前の確認点です。

**採点のしかた**（セクション 5 でも同じ関数を使います）

- **形式**: JSON として読め、キーが `answer` と `needs_review` のちょうど 2 つで、型が文字列と真偽値であること。
- **内容**: `needs_review` が正解と一致し、かつ正解の語句（例: 「9 月 30 日」と「23:59」）が `answer` に含まれること。

内容の判定には `needs_review` の一致が必要です。そのため、キーが欠けている出力は、文面が正しくても内容は × になります。文面そのものは、表で自分の目で確かめてください。

`Qwen/Qwen3.5-0.8B` は画像も扱えるモデルですが、`AutoModelForCausalLM` で読み込むとテキスト部分だけが使われます。

**見るところ**

- 1 つ目のセル: `重みのメモリ(GB): 3.01` のように表示されます。モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。
- 2 つ目のセル: 1 行の要約（`format_rate`、`content_rate` など）と、21 問の出力の表。評価には 1 分前後かかります。学習前は、多くの出力で `needs_review` が欠けているはずです。

**出ても問題ない警告**

- `You are sending unauthenticated requests to the HF Hub`: Hugging Face にログインせずにモデルを取りに行っている、という案内です。公開モデルはログインしなくてもダウンロードできます。ログインが要るのは次の場合で、モデルの大きさでは決まりません。
  - 利用規約への同意が必要なモデル（gated model。Llama や Gemma など）や、非公開のリポジトリを使うとき。
  - 自分のモデルやデータを Hub にアップロードするとき。
  - 短時間に大量のリクエストを送り、回数の上限に当たるとき。上限はログインしていないと IP アドレスごとに 5 分あたり 3,000 回のファイル取得（2025 年 9 月時点の公表値。[Hub の rate limits](https://huggingface.co/docs/hub/rate-limits)）で、ログインすると上がります。
  
  このノートブックのモデルは公開されていて同意も不要です。読み込みで送るリクエストは多くても数十回なので、上限には届きません。もし上限に当たって `429 Too Many Requests` が出ても、ライブラリが待ち時間を読み取って自動で再試行します。ログインが必要になったときは、Colab の左側の鍵アイコン（シークレット）に Hugging Face のトークンを `HF_TOKEN` という名前で登録し、このノートブックからのアクセスを許可すると、自動で使われます。
- `... is falling back to its reference PyTorch implementation ...`: このモデルの一部の層（線形注意）に対する高速な実装が入っていないので、遅いが正しい実装で動くという案内です。4 行ほど出ます。

**ここで読んでおくこと**: 表を見て、「どのキーが欠けているか」「文面は FAQ と合っているか」を確認します。この表はセクション 5 で学習後の出力と並べて見ます。出力の原文は `/content/qlora_lab/eval_before.json` にも保存されています（左のファイル一覧から開けます）。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
set_seed(CONFIG["seed"])

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 形式はプロンプトで指示している。モデルがこれを守れるかが学習前の確認点
SYSTEM = "あなたは架空の訓練用セキュリティ監視 FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。FAQ に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。"

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
# 既定の512トークンで教師データが切れないことを、学習前に確認する。
lengths = []
for ex in train + valid:
    text = tok.apply_chat_template(build_messages(ex) + [{"role": "assistant", "content": ex["target"]}], tokenize=False, enable_thinking=False)
    lengths.append(len(tok(text, add_special_tokens=False)["input_ids"]))
print("学習・検証データの最大トークン数:", max(lengths))
if max(lengths) > CONFIG["max_seq_len"]:
    raise ValueError("教師データがmax_seq_lenを超えます。FAQ/質問を短くして最初から実行してください。")


In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def check_format(text: str):
    '''JSON として読め、必須キーと型が合うか。失敗理由も返す'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''内容の合否: needs_review が正解と一致し、必須語句を answer に含む'''
    if not isinstance(obj, dict): return False
    if not isinstance(obj.get("answer"), str) or type(obj.get("needs_review")) is not bool: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''test の 21 問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 形式が壊れていても JSON らしい部分が読めるなら、内容だけは別に採点する
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は FAQ と合っているかを読む
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

## 4. LoRA 学習（T4 で 4〜5 分）

### LoRA と QLoRA の違い

- **LoRA**: 基盤モデルの重みは固定し、一部の層に追加した小さな行列（adapter）だけを学習します。
- **QLoRA**: 基盤モデルの重みを 4bit に量子化して保持し、その上で同じように adapter だけを学習します。adapter 自体は量子化しません。

QLoRA の効果は、基盤モデルの重みを保持するメモリが減ることです。16bit の約 4 分の 1 になるので、そのままでは GPU に載らない大きさのモデルでも学習できます（70 億パラメータなら 16bit で約 14GB、4bit で約 3.5GB）。

| 観点 | LoRA（量子化なし） | QLoRA（4bit） |
| --- | --- | --- |
| 重みのメモリ | 大きい | 小さい（16bit の約 1/4） |
| 基盤モデルの出力 | 元のまま | 量子化の誤差で変わることがある。影響の大きさはモデルによって違う |
| 学習の速さ | 基準 | 速くはならない。4bit の重みを計算のたびに戻すので、遅くなることが多い |
| 依存ライブラリ | peft | peft に加えて bitsandbytes（CUDA の GPU が前提） |

QLoRA を実装するときの注意:

- 読み込み時に `BitsAndBytesConfig(load_in_4bit=True, ...)` を渡し、LoRA を付ける前に `prepare_model_for_kbit_training(model)` を通します。
- 保持精度（4bit）と計算精度（`bnb_4bit_compute_dtype`。16bit など）は別の設定です。
- 学習前後の比較は、同じ量子化条件の基盤モデルで行います。学習前を量子化なし、学習後を 4bit で測ると、量子化の影響と学習の影響が混ざります。
- adapter は学習時の基盤モデルに合わせて作られます。配布や再利用のときは、基盤モデルの ID・revision と量子化設定を一緒に記録します。
- 4bit での学習が勧められないモデルもあります。Qwen3.5 は量子化の誤差が通常より大きいとして、4bit 学習を勧めていない[学習ライブラリのガイド](https://unsloth.ai/docs/models/qwen3.5/fine-tune)があります。

この演習のモデルは量子化なしでも T4 に載るので、LoRA で学習します。モデルが GPU に載るなら LoRA、載らないなら QLoRA かより小さいモデル、という順で考えます。4bit で読み込んだ場合のメモリと出力の違いは、セクション 8（発展）で確かめられます。

### 学習の実行

**やること**: train の 14 件を chat 形式（system / user / assistant）に直し、LoRA を付けて 20 エポック（40 ステップ）学習します。学習した adapter は `adapter/` に保存します。

- `target_modules` は LoRA を差し込む層です。注意機構の射影（`q_proj` など）と MLP の射影（`gate_proj` など）を指定しています。このモデルは 24 層のうち 18 層が線形注意で、`q_proj` などを持つのは残りの 6 層だけです。MLP は全層にあります。
- バッチ 4 × 勾配累積 2 = 1 ステップ 8 件。train は 14 件なので 1 エポックは 2 ステップ（8 件と 6 件）で、20 エポックで 40 ステップです。各例をちょうど 20 回学習します。学習率の warmup と減衰（cosine）はこの 40 ステップに対して効きます。
- 学習後のモデルは変数 `tuned` に入ります。LoRA は基盤モデルの層に差し込まれるので、元の変数 `model` で生成しても adapter 込みの出力になります。学習前の出力が必要なときは、セクション 6 のように adapter を一時的に無効にします。

**見るところ**

- `trainable params: ... || all params: ... || trainable%: ...`: 学習するパラメータが全体の数 % だけであることが LoRA の特徴です。
- `Training Loss` の表（10 ステップごと）: 1.5 前後から始まり、40 ステップで 0.01〜0.02 まで下がります。下がらないまま終わった場合は、設定を変えていないか確かめてください。
- `学習時間: ... 秒 | ステップ数: 40 | エポック: 20.0`: T4 で 240〜290 秒です。ステップ数とエポックは、学習データ 14 件 × 20 周が上の計算どおりに実行されたことの確認です。

**出ても問題ない警告**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` は、終端トークンの設定を tokenizer に合わせたという案内です。

**うまくいかないとき**: 学習が途中で止まった、または学習できなかった場合は、下の 4b を読んでください。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
train_output = trainer.train()  # 完了後に実測を記録
training_seconds = time.time() - t0
if trainer.state.epoch is None or trainer.state.epoch < CONFIG["num_train_epochs"]:
    raise RuntimeError("学習が指定エポック数まで完了していません。")
EXECUTED_TRAINING = True
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

### 4b. 学習できなかった場合

ColabでGPUが確保できない場合は、時間を変えて試してください。
途中で中断した場合、残っているチェックポイントからの再開は講座の説明を参照してください。
完全にやり直す場合はセッションを再起動し、最初から実行します。

講師の参考結果は別の講座FAQでの実験です。独自FAQの結果として転記しないでください。
実行できなかった範囲は `answer.md` に記録し、講座の代替提出ルールについて運営に確認してください。


## 5. 学習後評価（学習前と同じ test・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 21 問・同じ生成設定で評価し、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**

- 比較表: `format_rate`（形式遵守率）と `content_rate`（内容正解率）が学習前からどう変わったか。
- `退行`: 学習前は形式も内容も通っていたのに、学習後に落ちた質問です。空のリストが期待値です。
- 質問ごとの表: 残った失敗を読み、どの型かを見分けてください。
  - 根拠なしの質問（正解の `needs_review` が `True`）に、FAQ の別項目を当てはめて断定した。
  - 根拠ありの質問（正解が `False`）に、「記載がありません」と確認要求で逃げた。

学習後の回答は、教師データの文（FAQ の原文）をほぼそのまま返す形になります。形式と文面は揃いますが、「FAQ に根拠があるか」の判断まで学習できたとは限りません。

seed を固定しても、GPU 上の学習は完全には再現しません。形式遵守率と内容正解率の割合はほぼ同じでも、どの質問が残るかは実行ごとに変わることがあります。隣の人と「残った質問」が違っても異常ではないので、失敗の型で比べてください。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 学習前は形式も内容も通ったのに、学習後に落ちた質問（退行）
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# 質問ごとの出力原文。形式（JSON・必須キー・型）と内容（needs_review の一致と必須語句）の判定を自分の目で確かめる
detail = pd.DataFrame([{"質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. 知識を問う質問（形式を学んでも知識は保証されない）

**やること**: 形式の実験とは別に、FAQ にも学習データにも無い事実を 3 問尋ねます（対照として FAQ にある質問を 1 問加えます）。望ましい応答は「確認が必要（`needs_review: true`）」です。

応答を 3 つに分類します。

| 分類 | 意味 |
| --- | --- |
| `format_error` | JSON の形式を守れていない（学習前に多い） |
| `needs_review` | 「記載がありません。確認が必要です」と返した（望ましい応答） |
| `asserted` | `needs_review: false` で言い切った。FAQ に無い質問に対してなら、根拠のない断定（創作）です |

**見るところ**: 学習前は形式違反になりやすく、学習後は「確認が必要」と「断定」の両方が出ることがあります。出力の原文も表に出すので、断定した場合に何を根拠にしたように見えるか（FAQ の別項目を流用しているか）を読んでください。「Fine-Tuning では知識は絶対に直らない」と先に結論にせず、今回のデータでの結果を報告します。

学習前の出力は、adapter を一時的に無効にして取っています（`tuned.disable_adapter()`）。

In [ ]:
# FAQにあるかはgoldへの質問文字列一致ではなく、明示した正解ラベルで判断する。
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # LoRA は基盤モデルの層に差し込まれるので、学習後は `model` で生成しても adapter 込みになる。学習前の出力は adapter を一時的に無効にして取る
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"質問": q, "FAQ にある": KNOWLEDGE_IN_FAQ[q], "学習前の分類": classify(before_raw), "学習前の出力": before_raw,
                 "学習後の分類": classify(after_raw), "学習後の出力": after_raw})
display(pd.DataFrame(knowledge_rows))

## 7. 実測の保存と解答生成

学習前後のJSON、実行条件、学習時間、loss、解答を保存します。
最後にダウンロードされるZIPを展開し、`lectures/02/` の結果ファイルを公開forkへ反映してください。
自動生成した解答の本人確認欄を埋め、特に否定・余分な断定・誤った語句を原文で確認します。


In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": EXECUTED_TRAINING,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])
import math
from datetime import datetime, timezone
if not EXECUTED_TRAINING:
    raise RuntimeError("学習が完了していません。")
loss_logs = [row["loss"] for row in trainer.state.log_history if "loss" in row]
result["training"] = {"seconds": training_seconds, "global_step": trainer.state.global_step,
    "last_logged_loss": loss_logs[-1] if loss_logs else None, "mean_loss": train_output.training_loss}
if not math.isfinite(float(train_output.training_loss)):
    raise RuntimeError("lossが有限値ではありません。実行ログを確認してください。")
result["execution"] = {"finished_at_utc": datetime.now(timezone.utc).isoformat(),
    "gpu": torch.cuda.get_device_name(0), "seed": CONFIG["seed"], "dataset": "fictional_soc_faq_v1"}
result["dataset"] = {"train": len(train), "valid": len(valid), "test": len(test), "faq_items": len(FAQ)}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")

# 以下は lectures/02/soc_report.py と同じ解答生成処理。
"""実行済みのresult/evalから第2回解答を生成する。学習結果を推測しない。"""
import json
from pathlib import Path

REPO_URL = "https://github.com/masahikoyr/lecture-ai-engineering-2026"


def md(value):
    return str(value).replace("|", "\\|").replace("\n", " ")


def render_answer(result, before_rows, after_rows):
    if result.get("executed_training_myself") is not True:
        raise ValueError("学習が完了した本人実行の結果が必要です。参考結果を流用しません。")
    before = {r["question"]: r for r in before_rows}
    after = {r["question"]: r for r in after_rows}
    if not before or before.keys() != after.keys() or len(before) != len(before_rows) or len(after) != len(after_rows):
        raise ValueError("学習前後の評価質問が一致していません。")
    lines = [
        "# 第2回 解答：架空のセキュリティ監視FAQへのLoRA適用", "",
        f"- 公開リポジトリURL：{REPO_URL}",
        "- 選択レベル：実装", "- 題材：架空のセキュリティ監視FAQ（実在の顧客・組織情報なし）",
        "- この文書はColab実行結果から自動生成。提出前に回答原文と照合し、末尾の本人確認欄を記入する。", "",
        "## 演習", "",
        "演習の仕組みを用い、宿題の独自題材で学習前後を比較した。以下はこの題材の結果であり、講師の参考結果ではない。",
        "- 使用モデル・設定：", "```json", json.dumps(result["config"], ensure_ascii=False, indent=2), "```",
        "- 自分で学習を実行したか：はい（executed_training_myself: true）。Colab上で学習・前後評価を実行。",
        "- 方法：LoRA（量子化なし）。生成設定・評価質問・採点関数は学習前後で共通。",
        "- 実行情報：", "```json", json.dumps(result.get("execution", {}), ensure_ascii=False, indent=2), "```",
        "- ライブラリ：", "```json", json.dumps(result["versions"], ensure_ascii=False, indent=2), "```", "",
        "| 指標 | 学習前 | 学習後 |", "| --- | ---: | ---: |",
    ]
    for key in ["n", "format_rate", "content_rate", "both_rate", "seconds"]:
        a, b = result["before"][key], result["after"][key]
        if key.endswith("rate"): a, b = f"{a:.2%}", f"{b:.2%}"
        lines.append(f"| {key} | {a} | {b} |")
    lines += ["", "secondsは各評価処理の時間であり、学習時間ではない。",
              f"- 学習時間：{result['training']['seconds']:.2f}秒",
              f"- 最終記録loss：{result['training']['last_logged_loss']}（最終ログ値。全学習平均とは区別）",
              f"- 全学習平均loss：{result['training']['mean_loss']}",
              f"- 完了ステップ数：{result['training']['global_step']}", "",
              "### 退行と残った失敗", ""]
    regressed = [q for q in before if before[q]["both_ok"] and not after[q]["both_ok"]]
    lines += ["退行（学習前は形式・内容が両方正解、学習後はどちらかが不正解）：",
              *([f"- {md(q)}" for q in regressed] or ["- この評価では該当なし。"]), "",
              "| 質問 | 失敗の型 | 学習後の出力原文 |", "| --- | --- | --- |"]
    failed = 0
    for r in after_rows:
        if r["both_ok"]: continue
        failed += 1
        try: obj = json.loads(r["raw"])
        except (ValueError, TypeError): obj = None
        if not r["format_ok"]: kind = "形式違反"
        elif r["gold_needs_review"] and obj["needs_review"] is False: kind = "根拠なしの質問に断定"
        elif not r["gold_needs_review"] and obj["needs_review"] is True: kind = "根拠ありの質問に確認要求"
        else: kind = "必須語句の不一致（原文も確認）"
        lines.append(f"| {md(r['question'])} | {kind} | {md(r['raw'])} |")
    if not failed: lines.append("| 該当なし | この21問での結果。一般的な安全性を保証しない | — |")
    lines += ["", "### 知識を問う追加質問", "",
              "| 質問 | FAQ内の根拠 | 学習前 → 学習後 | 学習後の原文 |", "| --- | --- | --- | --- |"]
    for r in result["knowledge"]:
        lines.append(f"| {md(r['質問'])} | {'あり（対照）' if r['FAQ にある'] else 'なし'} | {r['学習前の分類']} → {r['学習後の分類']} | {md(r['学習後の出力'])} |")
    lines += ["", "assertedはneeds_review=falseという形式上の分類。根拠ありの対照質問では、断定しただけで誤答とは判定しない。原文とFAQを照合する。", "",
              "## 宿題", "", "### 変更したデータと評価設計", "",
              "FAQ・GROUNDED・UNGROUNDED、およびtest専用質問と知識質問を独自題材に置換した。",
              "5項目は、不審メールの報告、調査ログの原本保全、重要度の判断、端末隔離の承認、担当者間の引継ぎ。",
              "JSONのキーanswer（文字列）とneeds_review（真偽値）の設計を維持した。",
              "根拠なしの質問には、FAQにない電話番号・保存日数・製品名・期限・コマンド等を選んだ。",
              "話題はFAQに近いが、値や手順を確定できない質問とし、一般知識で補完せず確認を求めるかを測る。",
              "学習14件（根拠あり10・なし4）、検証7件（あり5・なし2）、評価21件（あり15・なし6）。",
              "質問文は分割間で重複させない。ただし同一FAQを全分割で共有するため、未知の運用規則への汎化の検証ではない。",
              "モデルにはFAQと質問のみを入力し、評価時にtarget・goldは渡さない。", "",
              "### 結果と考察", "",
              "結果は上の比較表・退行・失敗例・知識質問に記載した。学習前後の差を、形式の変化と内容の変化に分けて判断する。",
              "LoRAは基盤モデルの重みを固定し、小さな追加行列を学習する方法である。",
              "今回のように出力形式や確認要求の応答例を揃えたい場合に適している。",
              "一方、更新の多い規則や案件固有の事実を重みに保持させ、最新性と正確性まで保証する用途には向かない。",
              "検索で最新の根拠を渡す方法と組み合わせ、更新情報の取得と出力形式の習得を分けて設計することが考えられる。",
              "本実験でもFAQを毎回プロンプトに含めている。評価の改善を、規則を重みに暗記した証拠とは解釈しない。",
              "また、content_rateはneeds_reviewの一致と必須語句の包含による採点で、否定や余分な断定まで網羅しない。",
              "1回の学習・21問・固定seedでの比較であり、複数seedでの再現性や実運用上の安全性は今後の検証課題である。", "",
              "### セキュリティ×AIへの応用", "",
              "調査支援AIではJSON形式の妥当性と、判断の根拠の妥当性を別々に確認する必要がある。",
              "needs_review=trueは責任者へ確認を回す目印として利用できるが、falseを自動実行の許可にはしない。",
              "実運用ではスキーマ検証、根拠文書の照合、権限管理、人による承認を組み合わせて検証する。",
              "本課題のFAQは学習用の架空規則であり、実際の組織で利用する運用手順ではない。", "",
              "### 本人確認（提出前に記入）", "",
              "- 表とresult.json・回答原文を照合した：未記入",
              "- 実際に変わった点・変わらなかった点の具体例と自分の気づき：未記入",
              "- AI利用：ChatGPT（Codex）でデータ・コード・解答の準備を支援。実行結果の確認は本人が行う。", "",
              "## 出典", "",
              "- 講座公式notebooks/lora_format_tuning.ipynbを基に改修：https://github.com/matsuolab/lecture-ai-engineering-2026",
              "- Qwen公式モデル：https://huggingface.co/Qwen/Qwen3.5-0.8B",
              "- Hugging Face PEFT：https://huggingface.co/docs/peft/main/en/conceptual_guides/lora", ""]
    return "\n".join(lines)


def write_answer(work):
    work = Path(work)
    result = json.loads((work / "result.json").read_text(encoding="utf-8"))
    before = json.loads((work / "eval_before.json").read_text(encoding="utf-8"))["rows"]
    after = json.loads((work / "eval_after.json").read_text(encoding="utf-8"))["rows"]
    (work / "answer.md").write_text(render_answer(result, before, after), encoding="utf-8")

write_answer(WORK)
print((WORK / "answer.md").read_text(encoding="utf-8"))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "answer.md", "valid.jsonl"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)

import zipfile
archive = Path("/content/AIEngineering2026_Lec02_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["result.json", "eval_before.json", "eval_after.json", "answer.md", "train.jsonl", "valid.jsonl", "test.jsonl", "gold.json"]:
        z.write(WORK / name, "lectures/02/" + name)
from google.colab import files
files.download(str(archive))
